# Exp 06: Augmentations v3

Card: [exp06_augmentations_v3.md](../../experiments/exp06_augmentations_v3.md)


In [ ]:
import dataclasses
import os

import albumentations as A
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import precision_recall_curve

from core.augment import V3, read_boxes
from core.config import CLASSES, IMG_SIZE, MEAN, ROOT, SEED, STD, TRAIN_DIR
from core.cv import VIP, CVResult, Config, CrossValidator
from core.data import read_train
from core.gradcam import GradCamFigures
from core.scores import Scores
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-augmentations-v3-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    PROBE = {
        "img_size": 64,
        "n_other": 20,
        "folds": [0],
        "num_workers": 0,
    }  # robustness test of exp04
    N_SHOW = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    PROBE = {}
    N_SHOW = 4
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp06_augmentations_v3" / "resnet18.toml", **OVERRIDES
)
# trained in earlier experiments, only read
BASELINES = {
    "baseline (exp04)": Config.from_toml(
        CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.toml"
    ),
    "lecture (exp03)": Config.from_toml(
        CONFIG_DIR / "exp03_lecture_augmentations" / "resnet18.toml"
    ),
    "v2 (exp05)": Config.from_toml(
        CONFIG_DIR / "exp05_augmentations_v2" / "resnet18.toml"
    ),
}
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp06_augmentations_v3"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; color follows entity across figures
MUTED = "#898781"
SERIES = {
    "baseline (exp04)": "#eb6834",
    "v3": "#2a78d6",
    "v2 (exp05)": "#1baf7a",
    "lecture (exp03)": "#eda100",
}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## v3 draws


In [ ]:
N_DRAWS = 6
train_df = read_train()
only_cat = train_df[CLASSES].eq([1, 0, 0, 0, 0]).all(axis=1)
picks = {cls: train_df[train_df[cls] == 1] for cls in VIP} | {
    "Cat only": train_df[only_cat]
}
image_ids = {
    name: rows.sample(1, random_state=SEED)["image_id"].item()
    for name, rows in picks.items()
}
images = {
    name: np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg")
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )
    for name, image_id in image_ids.items()
}
boxes = read_boxes()  # Grad-CAM pseudo-boxes, needed by BBoxSafeRandomCrop
BOX = SERIES["v3"]
tfms = V3(IMG_SIZE).compose()
show = A.Compose(
    tfms.transforms[:-2], bbox_params=tfms.processors["bboxes"].params
)  # without Normalize, ToTensorV2


def draw(name):
    """(image, boxes, box labels); i = 0 original, else one v3 draw. Boxes follow every transform."""
    image_boxes = boxes.get(image_ids[name], [])
    bboxes = np.array([b for _, b in image_boxes]).reshape(-1, 4)
    out = show(
        image=images[name], bboxes=bboxes, box_labels=[c for c, _ in image_boxes]
    )
    return out["image"], out["bboxes"], out["box_labels"]


def plot_boxes(ax, image, bboxes, labels):
    h, w = image.shape[:2]
    for (x0, y0, x1, y1), cls in zip(bboxes, labels):
        ax.add_patch(
            plt.Rectangle(
                (x0 * w, y0 * h),
                (x1 - x0) * w,
                (y1 - y0) * h,
                fill=False,
                ec=BOX,
                lw=1.5,
            )
        )
        ax.text(
            x0 * w + 2,
            y0 * h + 2,
            cls,
            color="white",
            fontsize=7,
            va="top",
            bbox=dict(fc=BOX, ec="none", pad=1),
        )


fig, axes = plt.subplots(
    len(images), N_DRAWS + 1, figsize=(2.1 * (N_DRAWS + 1), 2.4 * len(images))
)
for (name, image), row_axes in zip(images.items(), axes):
    for i, ax in enumerate(row_axes):
        if i == 0:
            image_boxes = boxes.get(image_ids[name], [])
            shown, bboxes, labels = (
                image,
                [b for _, b in image_boxes],
                [c for c, _ in image_boxes],
            )
        else:
            shown, bboxes, labels = draw(name)
        ax.imshow(shown)
        plot_boxes(ax, shown, bboxes, labels)
        ax.set_axis_off()
        if name == VIP[0]:
            ax.set_title("Original" if i == 0 else f"Draw {i}", fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        name,
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {f"{PREFIX}_draws.png": fig}
plt.show()

## Robustness of exp04


In [ ]:
# Validation plus one transform at p = 1; seeded so every run sees same draws
PROBES = {
    "clean": [],
    "horizontal flip": [A.HorizontalFlip(p=1)],
    "vertical flip": [A.VerticalFlip(p=1)],
    "rotate 90": [A.RandomRotate90(p=1)],
    "darker": [
        A.OneOf(
            [
                A.RandomBrightnessContrast(
                    brightness_limit=(-0.3, -0.1), contrast_limit=0.2
                ),
                A.RandomGamma(gamma_limit=(140, 200)),  # gamma > 1 darkens
            ],
            p=1,
        )
    ],
    "blur": [A.GaussianBlur(blur_limit=(5, 7), p=1)],
    "JPEG 50": [A.ImageCompression(quality_range=(50, 50), p=1)],
    "gray": [A.ToGray(p=1)],
    "coarse dropout": [
        A.CoarseDropout(
            num_holes_range=(3, 8),
            hole_height_range=(0.05, 0.15),
            hole_width_range=(0.05, 0.15),
            fill="random_uniform",
            p=1,
        )
    ],
}
probe_cfg = dataclasses.replace(BASELINES["baseline (exp04)"], **PROBE)
size = probe_cfg.img_size
robust = {}
for name, extra in PROBES.items():
    tfms = A.Compose(
        [A.Resize(size, size), *extra, A.Normalize(MEAN, STD), ToTensorV2()], seed=SEED
    )
    fold_ap = []
    for fold in probe_cfg.folds:
        probs, y = probe_cfg.predict_val(fold, tfms)
        fold_ap.append(Scores(y, probs).ap())
    robust[name] = pd.DataFrame(fold_ap).mean()
robust = pd.DataFrame(robust).T
robust_drop = (robust - robust.loc["clean"]).add_suffix(" drop")
pd.concat([robust["mAP"], robust_drop], axis=1).round(3)

## Training


In [ ]:
results = {name: CVResult.load(cfg) for name, cfg in BASELINES.items()}

In [ ]:
results["v3"] = CrossValidator(CONFIG).run()

## Training curves


In [ ]:
CURVES = ["baseline (exp04)", "v3"]
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name in CURVES:
        r = results[name]
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(CONFIG.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="augmentations", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_training.png"] = fig
plt.show()

## AP comparison


In [ ]:
fold_ap_df = pd.concat(
    {name: results[name].fold_ap for name in CURVES}, names=["augmentations"]
)
ap = fold_ap_df.groupby("augmentations", sort=False).mean()
ap.round(3)

In [ ]:
# difference to exp04 counts only if it has same sign on both folds
diff = (results["v3"].fold_ap - results["baseline (exp04)"].fold_ap).T.add_prefix(
    "diff fold "
)
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Precision-recall curves


In [ ]:
PR = ["baseline (exp04)", "v2 (exp05)", "lecture (exp03)", "v3"]
fig, axes = plt.subplots(1, len(VIP), figsize=(4.2 * len(VIP), 4), sharey=True)
for ax, cls in zip(axes, VIP):
    for name in PR:
        oof = results[name].oof
        precision, recall, _ = precision_recall_curve(oof[cls], oof[f"p_{cls}"])
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=name,
        )
    ax.set(title=cls, xlabel="recall", xlim=(0, 1.02), ylim=(0, 1.02))
axes[0].set_ylabel("precision")
axes[-1].legend(loc="lower left", title="augmentations", alignment="left")
fig.suptitle(f"Out-of-fold rows of folds {CONFIG.folds}", color="#52514e")
fig.tight_layout()
figures[f"{PREFIX}_precision_recall.png"] = fig
plt.show()

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_table(robust.reset_index(names="probe"), "robustness_exp04.json")
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)